## AstroCoffee grind sandbox

The ArXiV threw us a loop on 2024 Feb 1 and made a substantial change to their RSS feed.  We have to 
now rewrite the entire parser.

Awesome.

This sandbox notebook looks at using Python to generate the coffee paper database (pdb) file that was
created originally with the `grind.pl` Perl script.

We use the python modules `requests` and `BeautifulSoup` that come with Anaconda python to make a more
modern (and way nicer) parser.



In [1]:
import sys
import os
import re
import numpy as np
from datetime import datetime, timezone

# XML parsing

import requests
from bs4 import BeautifulSoup

# HTML display inside the notebook

from IPython.display import HTML


## make an html page like produced by grind.

In [7]:
versionID = '3.0.5'
versionDate = '2026-03-08'

# ArXiV RSS Feed for astro-ph

rssFeed = 'http://arxiv.org/rss/astro-ph'

# isoTime

now = datetime.now()
calDate = now.strftime('%A %B %e')
print(f"Today's local date is {calDate}")

utc = datetime.now(timezone.utc)
isoDate = utc.strftime('%Y%b%d')
print(f"Today's UTC date is {isoDate}")

# brew log relative URL

brewLog = f'/Coffee/brew.log'

# daily paper database file

coffeePot = './'  # else /var/www/CoffeePot

dbFile = f'{coffeePot}/astro-ph.pdb'

# maximum number of authors to list before switching to et al.

maxAuth = 10


Today's local date is Tuesday March 10
Today's UTC date is 2026Mar10


### read the RSS feed

In [3]:
r = requests.get(rssFeed)
soup = BeautifulSoup(r.text,'xml')
papers = soup.findAll('item')

numPapers = len(papers)

### First check, did we get anything?

In [4]:
# We didn't get anything

if numPapers == 0:
    print("got nuthin")

    gotNothing = f"""Content-Type: text/html

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN">
<HTML>
<HEAD>
<TITLE>Astro Coffee - No Papers Found</TITLE>
<meta http-equiv="content-type" content="text/html;charset=iso-8859-1">
</HEAD>
<BODY>

<h1>Error: No papers listed in the arXiv RSS feed</h1>

The arXiv RSS feed channel could not be retrieved or has no papers listed. 
Please let Rick Pogge know you had problems, and if he is around hopefully
it will be solved before coffee time. If not, be ready to do it by hand the
old fashioned way...

<hr>
grind.py v{versionID} [{versionDate}]
</BODY>
</HTML>
"""

    print(gotNothing)
    HTML(gotNothing)
    #sys.exit(1)
else:
    print(f'Found {numPapers} today')

Found 122 today


### Parse the RSS feed response into data we need

 * `paperID` = paper's arXiv ID (e.g., 2401.17318)
 * `absURL` = URL of the arXiv full abstract page
 * `pdfURL` = URL of the paper PDF file
 * `paperName` = unique paper identifier (e.g., arxiv:2401.17318)
 * `paperType` = type of submission (NEW, CROSS, REPLACE)
 * `paperTitle` = full text of the paper title.
 * `paperAuths` = abbreviated paper list
 * `paperAbst` = full text of the paper abstract

These get stored in a paper database file for use by CGI scripts downstream.  This makes sure we only 
make 1 (or at least very few) queries of the RSS feed so an over-enthusiastic user doesn't hammer the
arxiv feed and get tagged (incorrectly) as a bot.

In [11]:
# posting date derived from the pubData element

pubDate = soup.findAll('pubDate')
pubBits = pubDate[0].text.split(' ')
postingDate = f'{pubBits[3]} {pubBits[2]} {pubBits[1]}'

# temporary lists because of some infelicities in the feed

arxivIDs = []
titles = []
absLinks = []
pdfLinks = []
absNames = []
abstracts = []
authors = []
subTypes = []

for paper in papers:
    titles.append(paper.find('title').text)
    link = paper.find('link').text
    
    # pick apart the link into the bits we need
    
    protocol,url = link.split('//')
    urlBits = url.split('/')
    paperID = urlBits[-1]
    arxivIDs.append(paperID)
    absLinks.append(f'{protocol}//arxiv.org/abs/{paperID}')
    pdfLinks.append(f'{protocol}//arxiv.org/pdf/{paperID}.pdf')
    absNames.append(f'arXiv:{paperID}')
    
    # submission type
    
    subTypes.append(paper.find('arxiv:announce_type').text.upper())

    # paper abstract
    
    abstracts.append(paper.find('description').text)

    # authors - broken into multiple elements, merge...

    rawAuth = paper.find('dc:creator')
    pAuth = re.sub( "<\/?dc:creator>", "", rawAuth.text).split(',')

    for auth in pAuth:
        if auth == pAuth[0]:
            fullAuth = pAuth[0]
        else:
            fullAuth = f'{fullAuth}, {auth}'
    if len(pAuth) > 1:
        authStr = f'{pAuth[0]}, et al. [{len(pAuth)-1} co-authors]'
    else:
        authStr = f'{pAuth}'
        fullAuth = pAuth

    
    if len(pAuth) > maxAuth or len(pAuth) == 1:
        authors.append(authStr)
    else:
        authors.append(fullAuth)

    
# numpy arrays of our primary data

paperID = np.array(arxivIDs)
absURL = np.array(absLinks)
pdfURL = np.array(pdfLinks)
paperName = np.array(absNames)
paperType = np.array(subTypes)
paperTitle = np.array(titles)
paperAuths = np.array(authors)
paperAbst = np.array(abstracts)

# accounting

numPapers = len(papers)

# number of new submissions

iNew = np.where(paperType=='NEW')[0]
numNew = len(iNew)

# number of cross-posted papers

iCross = np.where(paperType=='CROSS')[0]
numCross = len(iCross)

# number of replacements

iUpdated = np.where((paperType=='REPLACE') | (paperType=='REPLACE-CROSS'))[0]
numUpdated = len(iUpdated)

# quick report

print(f"astro-ph RSS Feed summary:")
print(f"  Publication date tag:\n     {pubDate[0]}")
print(f"  Found {numPapers} papers on astro-ph today:")
print(f"     {numNew} new")
print(f"     {numCross} cross-posted")
print(f"     {numUpdated} replacements")

# last step before we go, delete the old database file

if os.path.exists(dbFile):
    os.unlink(dbFile)

pbd = open(dbFile,'w')
for i in iNew:
    dbStr = f'{paperID[i]}|astro-ph|{absURL[i]}|{pdfURL[i]}|{paperName[i]}|{paperType[i]}|{paperTitle[i]}|{paperAuths[i]}|{paperAbst[i]}'
    pbd.write(f'{dbStr}\n')
for i in iCross:    
    dbStr = f'{paperID[i]}|astro-ph|{absURL[i]}|{pdfURL[i]}|{paperName[i]}|{paperType[i]}|{paperTitle[i]}|{paperAuths[i]}|{paperAbst[i]}'
    pbd.write(f'{dbStr}\n')
for i in iUpdated:
    dbStr = f'{paperID[i]}|astro-ph|{absURL[i]}|{pdfURL[i]}|{paperName[i]}|{paperType[i]}|{paperTitle[i]}|{paperAuths[i]}|{paperAbst[i]}'
    pbd.write(f'{dbStr}\n')

pbd.close()

astro-ph RSS Feed summary:
  Publication date tag:
     <pubDate>Tue, 10 Mar 2026 00:00:00 -0400</pubDate>
  Found 122 papers on astro-ph today:
     73 new
     12 cross-posted
     37 replacements


### Build the 'grind' webpage


In [6]:
htmlText = ''

# HTML compliant header and page title

htmlHead = """Content-Type: text/html

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN">
<HTML>
<HEAD>
<TITLE>Astro Coffee Step 1: Grind...</TITLE>
<meta http-equiv="content-type" content="text/html;charset=iso-8859-1">
</HEAD>
<BODY BGCOLOR="#FFFFFF">
<p>
<center><h1><cite>The Daily Grind</cite></h1></center>
"""
#print(htmlHead)
htmlText += htmlHead

# Notices and Instructions

instructions = f"""<h2>Instructions</h2>
<p>This set of web forms is used to select a subset of the most recent 
astro-ph abstracts for discussion at Astro Coffee. The procedure is as follows: 
<dl> 
<p><dd><b>Step 1</b>: Select astro-ph papers using the form below. 
<p><dd><b>Step 2</b>: Review your selections. 
<p><dd><b>Step 3</b>: Post today's edition of the <a href="/Coffee/coffee.html"><i>Daily Brew</i></a>, 
and update the <a href="/Coffee/Archive/">AstroCoffee Archive</a>. 
</dl> 
<p>In general, only one person should run these scripts on a given 
day.  Check the <a href="{brewLog}">Brew Log</a> to see when this
was last done. 

<p>
<hr>

<h2>Step 1: Select Papers</h2> 

<p>Listed below are the abstracts in the latest mailing from
arXiv.org for astro-ph.  Note that if there was a problem with delivery of
today's abstract list by email, you may have to select abstracts by 
hand the old-fashioned way! 
"""

#print(instructions)
htmlText += instructions

# Build the selection form

formHeader = f"""<p>
<center><h3>astro-ph abstracts for {calDate}</h3></center>
{numPapers} Papers posted on <b>{postingDate}</b>
<p>
<form action="/cgi-bin/Coffee/brew.pl" method="POST">"""

#print(formHeader)
htmlText += formHeader

# New papers

newPapers = f"""
<h3>New Papers ({numNew}):</h3>
<table border=0>"""

#print(newPapers)
htmlText += newPapers

for i in iNew:
    entryStr = f"""<tr>
<td style="vertical-align:top"><input type="checkbox" name="abstract" value="{paperID[i]}"></td>
<td style="text-align:left"><b>{paperTitle[i]}</b><br>
{paperAuths[i]}<br>
<a href="{absURL[i]}">{paperName[i]}</a>
</td>
</tr>
<tr><td>&nbsp;</td><td>&nbsp;</td></tr>"""
    #print(entryStr)
    htmlText += entryStr
#print("</table>")
htmlText += '</table>'

# Cross-listings

if numCross > 0:
    crossPapers = f"""
<h3>Cross Listings ({numCross}):</h3>
<table border=0>"""

    #aprint(crossPapers)
    htmlText += crossPapers
    
    for i in iCross:
        entryStr = f"""<tr>
<td style="vertical-align:top"><input type="checkbox" name="abstract" value="{paperID[i]}"></td>
<td style="text-align:left"><b>{paperTitle[i]}</b><br>
{paperAuths[i]}<br>
<a href="{absURL[i]}">{paperName[i]}</a>
</td>
</tr>
<tr><td>&nbsp;</td><td>&nbsp;</td></tr>"""
        #print(entryStr)
        htmlText += entryStr
    #print("</table>")
    htmlText += '</table>'
    
# updated papers

if numUpdated > 0:
    repPapers = f"""
<h3>Replacements ({numUpdated}):</h3>
<table border=0>"""

    #print(repPapers)
    htmlText += repPapers
    
    for i in iUpdated:
        entryStr = f"""<tr>
<td style="vertical-align:top"><input type="checkbox" name="abstract" value="{paperID[i]}"></td>
<td style="text-align:left"><b>{paperTitle[i]}</b><br>
{paperAuths[i]}<br>
<a href="{absURL[i]}">{paperName[i]}</a>
</td>
</tr>
<tr><td>&nbsp;</td><td>&nbsp;</td></tr>"""
        #print(entryStr)
        htmlText += entryStr
        
    #print("</table>")
    htmlText += '</table>'
    
# Add the hidden keywords that pass along information needed by subsequent forms

hiddenKeys = f"""
<input type="hidden" name="NumAbstracts" value="{numPapers}">
<input type="hidden" name="AbstListing" value="{dbFile}">
<input type="hidden" name="ArchiveName" value="{isoDate}.html">
<input type="hidden" name="PostingDate" value="{postingDate}">
"""
#print(hiddenKeys)
htmlText += hiddenKeys

# Provide a space for the user to add arxiv papers from outside the
# astro-ph cross-listing space explicitly by number

# Provide a space for the user to add previous arxiv papers by arXiv number

bottomMatter = f"""
<h3>Previous arXiv Papers</h3>
Add links to previous arXiv papers by arXiv number (e.g., 2202.04273v2)
<b>one arXiv number line</b><br>
<textarea name="OldPapers" cols=24 rows=4></textarea>

<h3>Other Papers/Links</h3>
Add links by full URL (e.g., https://www.nature.com/articles/d41586-022-00425-8)
<b>one URL per line</b><br>
<textarea name="OtherPapers" cols=24 rows=4></textarea>

<input type="hidden" name="Printer" value="NONE">

<p>
<input type="SUBMIT" value="Submit Choices">  &nbsp; or &nbsp; 
<input type="RESET" value="Clear Choices">

</form>
<hr>
grind.py v{versionID} [{versionDate}]
</BODY>
</HTML>
"""

#print(bottomMatter)
htmlText += bottomMatter

# show it

#print(htmlText)
HTML(htmlText)

,"UK White Paper on Magnetohydrodynamic (MHD) seismology of solar and heliospheric plasmas Valery M. Nakariakov, David B. Jess, Andrew N. Wright, Timothy K. Yeoman, Thomas Elsden, James A. McLaughlin, Dmitrii Y. Kolotkov, Viktor Fedun, Robertus Erd\'elyi arXiv:2603.05571"
,
,"ENhanced Galactic Atmospheres With Arepo: Resolving the CGM at 200 pc with the ENGAWA Simulations Scott Lucchini, Cecilia Abramson, Cameron Hummels, Charlie Conroy, Lars Hernquist, Aaron Smith arXiv:2603.05584"
,
,"The TESS All-Sky Rotation Survey: Periods for 944,056 Stars Within 500 pc Andrew W. Boyle, Luke G. Bouma, Andrew W. Mann arXiv:2603.05586"
,
,"Predictions of Imminent Earth Impactors Discovered by LSST Ian Chow, Mario Juri\'c, R. Lynne Jones, Kathleen Kiker, Joachim Moeyens, Peter G. Brown, Aren N. Heinze, Jacob A. Kurlander arXiv:2603.05587"
,
,"Radiation GRMHD Models of Accretion onto Stellar-Mass Black Holes: III. Near-Eddington Accretion Lizhong Zhang, James M. Stone, Shane W. Davis, Yan-Fei Jiang, Patrick D. Mullen, Christopher J. White arXiv:2603.05588"
,
,"The Extremely Large Telescope Interferometer Francisco Prada, Enrique Perez, Sergio Fernandez-Acosta, Km Nitu Rai, Joel Sanchez-Bermudez arXiv:2603.05589"
